In [1]:
import sys
from pathlib import Path

from truststore import inject_into_ssl

inject_into_ssl()

sys.path.append(str(Path.cwd().parent))

import requests  # noqa: E402
from src.config.settings import IndexerSettings, SnowSettings  # noqa: E402

In [2]:
indexer_settings = IndexerSettings()
snow_settings = SnowSettings()
print(f"indexer_settings: {indexer_settings}")
print(f"snow_settings: {snow_settings}")

# proxies
HTTP_PROXY = indexer_settings.http_proxy
HTTPS_PROXY = indexer_settings.https_proxy
NO_PROXY = indexer_settings.no_proxy

# snow settings
snow_url = snow_settings.servicenow_url
snow_client_id = snow_settings.servicenow_client_id
# secret accessable through snow_settings.servicenow_client_secret

print(snow_settings.token_url)

indexer_settings: http_proxy='http://internet-proxy-client.muenchen.de:80' https_proxy='http://internet-proxy-client.muenchen.de:80' no_proxy='localhost,.svc.cluster.local' qdrant_url='https://snow-semantic-squirrel-test-qdrant.apps.test.capk.muenchen.de/' qdrant_api_key='' qdrant_timeout=100 collection_name='eakte-snow-kb' openai_embedding_model='text-embedding-3-large' openai_api_base='https://ki-proxy-test.muenchen.de' openai_api_key='sk-7PuBA_eFmJcZHETKfB4cQQ' embedding_timeout=10 embedding_max_retries=2 indexing_mode='hybrid' dense_vector_name='dense' sparse_vector_name='sparse' sparse_embedding_model='Qdrant/bm25' sparse_embedding_language='german' fastembed_cache_path='./model_cache' document_chunk_size=1000 document_chunk_overlap=200 indexing_batch_size=20 allow_empty_snapshot=False
snow_settings: servicenow_url='https://lhm.service-now.com/api/sn_km_api/knowledge/articles?kb=7ebcf5acc33a2210d05cf6fe05013198' servicenow_client_id='027496691e91432daa4713bfc99d9006' servicenow_cl

In [3]:
token_url = snow_settings.token_url
session = requests.Session()
session.proxies.update(snow_settings.proxies)
data = {
    "grant_type": "client_credentials",
    "client_id": snow_client_id,
    "client_secret": snow_settings.servicenow_client_secret.strip() if snow_settings.servicenow_client_secret else "",
    "scope": snow_settings.servicenow_oauth_scope,
}
headers = {"Content-Type": "application/x-www-form-urlencoded"}
print("Requesting ServiceNow access token")
resp = session.post(token_url, data=data, headers=headers, verify=True)
print(resp.status_code)

try:
    print(resp.json())
except ValueError:
    print(resp.text)
resp.raise_for_status()
token = resp.json().get("access_token")
if not token:
    raise RuntimeError("ServiceNow OAuth response did not contain an access_token")
print(token)

Requesting ServiceNow access token
200
{'access_token': 'sh72vyQWhAtg_51ZRglFj8001H5UKgrZJWNKxA9KNubK3zqmntGNJk55DPNmIbZq1uYOJ6Y6BeyPNxYSB5sHTw', 'scope': 'sn_km_api/knowledge.read', 'token_type': 'Bearer', 'expires_in': 1799}
sh72vyQWhAtg_51ZRglFj8001H5UKgrZJWNKxA9KNubK3zqmntGNJk55DPNmIbZq1uYOJ6Y6BeyPNxYSB5sHTw


In [4]:
def get_category(article):
    meta_description = article.get("meta_description", "")
    if "eakte-nutzer*innen" in meta_description["value"].lower():
        return "user"
    elif "eakte-fachadministrator*innen" in meta_description["value"].lower():
        return "admin"
    else:
        return "general"

In [5]:
session.headers.update({"Authorization": f"Bearer {token}"})
META_FIELDS = ",".join(
    [
        "kb_category",
        "kb_knowledge_base",
        "author",
        "workflow_state",
        "sys_created_on",
        "sys_updated_on",
        "valid_to",
        "sys_view_count",
        "keywords",
        "meta_description",
        "sys_attachment"
    ]
)

params = {
    "limit": snow_settings.servicenow_page_size,
    "fields": META_FIELDS,  # <-- add this (KM API param, not sysparm_fields)
}
data = session.get(snow_url, params=params, verify=True, proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY})
articles = data.json()["result"]["articles"]

for article in articles:
    print(article["number"], article["id"], list(article.keys()))
    # articles are split in generally two scpoes: for users and for admins. this is defined in the meta_description field
    # this is important for the vectordb, because we should be able to distinguish between the two scopes when searching for articles.
    # this enables us to provide more specific search results based on the user's role or needs.
    scope = get_category(article.get("fields", {}))
    print(article.get("fields"))  # <-- this is the check that matters

KB0026284 kb_knowledge:00e32ebbc33acf505c7f449dc00131ef ['link', 'id', 'title', 'snippet', 'score', 'number', 'fields']
{'kb_category': {'display_value': 'eAkte Ablage & Vorgangsbearbeitung', 'name': 'kb_category', 'label': 'Kategorie', 'type': 'reference', 'value': '6261ef0bc33eee1472c9716dc001310e'}, 'kb_knowledge_base': {'display_value': 'eAkte', 'name': 'kb_knowledge_base', 'label': 'Wissensdatenbank', 'type': 'reference', 'value': '7ebcf5acc33a2210d05cf6fe05013198'}, 'author': {'display_value': 'Jennifer Skurka', 'name': 'author', 'label': 'Autor', 'type': 'reference', 'value': '958b6f46db7a29d0266b0dcbd3961994'}, 'workflow_state': {'display_value': 'Veröffentlicht', 'name': 'workflow_state', 'label': 'Workflow-Status', 'type': 'workflow', 'value': 'published'}, 'sys_created_on': {'display_value': '26.08.2026 11:51:03', 'name': 'sys_created_on', 'label': 'Erstellt', 'type': 'glide_date_time', 'value': '2026-08-26 09:51:03'}, 'sys_updated_on': {'display_value': '26.08.2026 11:51:14

In [6]:
fields = ",".join(
    [
        "sys_id",
        "number",
        "short_description",
        "text",
        "kb_knowledge_base",
        "kb_category",
        "topic",
        "category",
        "workflow_state",
        "published",
        "valid_to",
        "sys_created_on",
        "sys_updated_on",
        "author",
        "sys_view_count",
        "meta_description",
        "keywords",
        "sys_attachment",
    ]
)

article_url = "https://lhm.service-now.com/api/sn_km_api/knowledge/articles/{}"
full_articles = []

for article in articles:
    r = session.get(
        article_url.format(article["id"].split(":")[1]),
        params={
            "sysparm_fields": fields,
            "sysparm_display_value": "all",  # resolve reference fields
        },
        verify=True,
        proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY},
    )
    r.raise_for_status()
    full_articles.append(r.json()["result"])

print(len(full_articles))

100


In [7]:
for i in range(len(full_articles)):
    print(full_articles[i].keys())

dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])
dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_con

In [8]:
from src.loaders.snow_loader import SnowLoader  # noqa: E402

loader = SnowLoader(config=snow_settings)

docs = loader.load_documents()

In [9]:
from pprint import pprint  # noqa: E402

for doc in docs:
    print(doc.metadata['attachments'])
pprint(docs[0].metadata["attachments"])
print(len(docs))

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
103


## Attachment POC

`display_attachments` is a display flag, not a list of attachment records. The Knowledge API may nevertheless include attachment URLs in `content` or `embedded_content`; this POC extracts those links and any attachment sys_ids from the article payload. This is useful for linked/embedded files, but the Knowledge API response cannot reliably enumerate ordinary record attachments when it does not expose their attachment sys_ids.

In [10]:
# Quick POC fix: display_attachments is a boolean; embedded_content contains the files.
from urllib.parse import urljoin, urlsplit

attachment_instance_url = f"{urlsplit(snow_url).scheme}://{urlsplit(snow_url).netloc}/"

articles_with_attachments = [
    {**article, "attachments": article.get("embedded_content") or []}
    for article in full_articles
]

embedded_attachments = [
    {
        **attachment,
        "article_number": article.get("number"),
        "article_sys_id": article.get("sys_id"),
        "api_file_url": urljoin(
            attachment_instance_url,
            f"api/now/attachment/{attachment['sys_id']}/file",
        ),
    }
    for article in articles_with_attachments
    for attachment in article["attachments"]
]

print(f"Attachments exposed through embedded_content: {len(embedded_attachments)}")
for attachment in embedded_attachments:
    print(attachment)

Attachments exposed through embedded_content: 614
{'sys_id': '0896e27fc37acf505c7f449dc00131a9', 'file_name': '9.png', 'size_bytes': '174696', 'state': 'available', 'article_number': 'KB0026284', 'article_sys_id': '00e32ebbc33acf505c7f449dc00131ef', 'api_file_url': 'https://lhm.service-now.com/api/now/attachment/0896e27fc37acf505c7f449dc00131a9/file'}
{'sys_id': '0b766a3fc37acf505c7f449dc001311c', 'file_name': '7.png', 'size_bytes': '145355', 'state': 'available', 'article_number': 'KB0026284', 'article_sys_id': '00e32ebbc33acf505c7f449dc00131ef', 'api_file_url': 'https://lhm.service-now.com/api/now/attachment/0b766a3fc37acf505c7f449dc001311c/file'}
{'sys_id': '0ba6ae3fc37acf505c7f449dc00131d9', 'file_name': '12.png', 'size_bytes': '174496', 'state': 'available', 'article_number': 'KB0026284', 'article_sys_id': '00e32ebbc33acf505c7f449dc00131ef', 'api_file_url': 'https://lhm.service-now.com/api/now/attachment/0ba6ae3fc37acf505c7f449dc00131d9/file'}
{'sys_id': '1e56a27bc37acf505c7f449dc

In [13]:
# Download the first three POC attachments with the authenticated API session.
import hashlib
from pathlib import Path

download_dir = Path("attachment_downloads/pngs")
download_dir.mkdir(parents=True, exist_ok=True)

downloaded_attachments = []
for attachment in embedded_attachments[::]:
    if attachment["file_name"].lower().endswith((".pdf")):
        continue
    print(f"Downloading attachment {attachment['file_name']} from article {attachment['article_number']}")
    attachment_sys_id = attachment["sys_id"]
    safe_file_name = Path(attachment["file_name"]).name
    destination = download_dir / f"{attachment_sys_id}_{safe_file_name}"
    digest = hashlib.sha256()
    size_bytes = 0

    with session.get(
        attachment["api_file_url"],
        headers={"Accept": "*/*"},
        stream=True,
        timeout=180,
        verify=snow_settings.servicenow_verify_ssl,
    ) as response:
        response.raise_for_status()
        with destination.open("wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                output_file.write(chunk)
                digest.update(chunk)
                size_bytes += len(chunk)

    result = {
        "sys_id": attachment_sys_id,
        "path": str(destination.resolve()),
        "size_bytes": size_bytes,
        "sha256": digest.hexdigest(),
    }
    downloaded_attachments.append(result)
    print(result)

#assert len(downloaded_attachments) == min(3, len(embedded_attachments))

{'sys_id': '0896e27fc37acf505c7f449dc00131a9', 'path': 'C:\\Users\\sebastian.berger\\projects\\opensource-projects\\snowman\\indexer\\notebooks\\attachment_downloads\\pngs\\0896e27fc37acf505c7f449dc00131a9_9.png', 'size_bytes': 174696, 'sha256': 'dd14da22c5e46341a7837059f07d0e3362a9c5d071bbdb267f109e1c349ed352'}
{'sys_id': '0b766a3fc37acf505c7f449dc001311c', 'path': 'C:\\Users\\sebastian.berger\\projects\\opensource-projects\\snowman\\indexer\\notebooks\\attachment_downloads\\pngs\\0b766a3fc37acf505c7f449dc001311c_7.png', 'size_bytes': 145355, 'sha256': 'ab3603af5715222228f62eebf4883d4716e95fd0ccb818c50a5d538e5486e2ec'}
{'sys_id': '0ba6ae3fc37acf505c7f449dc00131d9', 'path': 'C:\\Users\\sebastian.berger\\projects\\opensource-projects\\snowman\\indexer\\notebooks\\attachment_downloads\\pngs\\0ba6ae3fc37acf505c7f449dc00131d9_12.png', 'size_bytes': 174496, 'sha256': '515308576577a3f6a40fce6128c955e8df6de12c7b99fd5a27172d9a947ad0c1'}
{'sys_id': '1e56a27bc37acf505c7f449dc00131b6', 'path': 'C

### Production follow-up: enumerate `sys_attachment`

The quick fix above is limited to files that the Knowledge API exposes in `embedded_content`. A robust implementation should enumerate attachment records independently, as the public `snowloader` project does:

1. Query `GET /api/now/table/sys_attachment` with `sysparm_query=table_name=kb_knowledge^table_sys_idIN<article sys_ids>`. Batch the article IDs if the encoded query becomes too long.
2. Request at least `sys_id,file_name,content_type,size_bytes,table_name,table_sys_id,sys_created_on,sys_updated_on`.
3. Use deterministic ordering ending in `ORDERBYsys_id`, paginate by `sysparm_limit` and `sysparm_offset`, and continue until an empty page. A short page is not necessarily the last page because ServiceNow ACL filtering can shorten responses.
4. Join each attachment back to its article with `table_sys_id`. Confirm the instance uses `kb_knowledge` as `table_name`; extended or custom KB tables may differ.
5. Download bytes from `GET /api/now/attachment/{attachment_sys_id}/file` with the authenticated API session and `Accept: */*`; do not use the browser-facing `sys_attachment.do` URL, which may redirect to `navpage.do`.
6. Ensure the OAuth principal has read ACL access to `sys_attachment` and the attachment file endpoint. Add retries, timeouts, and an optional size limit for eager downloads.

This enumerates ordinary record attachments even when they are absent from article HTML and `embedded_content`. It returns metadata/raw bytes only; PDF, Office, image, audio, or video content still needs a separate parsing pipeline before indexing.

In [12]:
import re
from html.parser import HTMLParser
from urllib.parse import parse_qs, urljoin, urlsplit

instance_url = f"{urlsplit(snow_url).scheme}://{urlsplit(snow_url).netloc}/"
sys_id_pattern = re.compile(r"[0-9a-f]{32}", re.IGNORECASE)

class ArticleAttachmentParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.references = []

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        attribute = "href" if tag == "a" else "src" if tag in {"img", "source"} else None
        raw_url = attrs.get(attribute) if attribute else None
        if raw_url and ("sys_attachment.do" in raw_url or "/api/now/attachment/" in raw_url):
            self.references.append((tag, raw_url))

def attachment_references(article):
    parser = ArticleAttachmentParser()
    parser.feed(article.get("content") or "")
    references = []
    article_api_url = urljoin(
        instance_url,
        f"api/sn_km_api/knowledge/articles/{article['sys_id']}",
    )
    article_ui_path = f"?id=kb_article_view&sys_kb_id={article['sys_id']}"
    for tag, raw_url in parser.references:
        browser_url = urljoin(instance_url, raw_url)
        parsed = urlsplit(browser_url)
        query_sys_id = parse_qs(parsed.query).get("sys_id", [None])[0]
        path_sys_ids = sys_id_pattern.findall(parsed.path)
        attachment_sys_id = query_sys_id or (path_sys_ids[-1] if path_sys_ids else None)
        references.append({
            "article_number": article.get("number"),
            "article_api_url": article_api_url,
            "article_ui_path": article_ui_path,
            "kind": "download_link" if tag == "a" else "inline_image",
            "attachment_sys_id": attachment_sys_id,
            "browser_url": browser_url,
            "api_file_url": (
                urljoin(instance_url, f"api/now/attachment/{attachment_sys_id}/file")
                if attachment_sys_id else None
            ),
            "raw_artilcle": article,
        })
    return list({item["browser_url"]: item for item in references}.values())

references = [ref for article in full_articles for ref in attachment_references(article)]
download_links = [ref for ref in references if ref["kind"] == "download_link"]
inline_images = [ref for ref in references if ref["kind"] == "inline_image"]

print(f"Actual <a href> attachment links: {len(download_links)}")
print(f"Inline attachment images (not file links): {len(inline_images)}")
for link in download_links:
    print(link)

if download_links:
    probe = session.get(
        download_links[0]["browser_url"],
        allow_redirects=False,
        verify=snow_settings.servicenow_verify_ssl,
        proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY},
    )
    print("First link response:", probe.status_code, probe.headers.get("Content-Type"), probe.headers.get("Location"))

print("display_attachments values:", {repr(a.get("display_attachments")) for a in full_articles})

Actual <a href> attachment links: 13
Inline attachment images (not file links): 598
{'article_number': 'KB0025406', 'article_api_url': 'https://lhm.service-now.com/api/sn_km_api/knowledge/articles/47d66008475247144511c2da116d43e1', 'article_ui_path': '?id=kb_article_view&sys_kb_id=47d66008475247144511c2da116d43e1', 'kind': 'download_link', 'attachment_sys_id': '03d6a008475247144511c2da116d4306', 'browser_url': 'https://lhm.service-now.com/sys_attachment.do?sys_id=03d6a008475247144511c2da116d4306', 'api_file_url': 'https://lhm.service-now.com/api/now/attachment/03d6a008475247144511c2da116d4306/file', 'raw_artilcle': {'content': '<p>Im Mai 2026 fand die Informationsveranstaltung zum F&uuml;hrungskr&auml;fte-Delta statt. Die Informationsveranstaltung richtet sich an F&uuml;hrungskr&auml;fte, die sich bereits in der eAkte auskennen.</p>\r\n<p>Der Termin soll der Orientierung dienen und einen &Uuml;berblick schaffen, was sich durch die Einf&uuml;hrung des Produktstandards (MUCS26) am 12. Ok

In [13]:
https://enablenow.muenchen.de/sen/pub/p_eakte/project/PR_2A51E3CE01EA8BBE/lhm_training.pdf

SyntaxError: invalid syntax (3674171786.py, line 1)